In [1]:
# Generate Bloomberg Excel workbooks
#
# Run after 01_prepare_bloomberg_identifiers.ipynb. This notebook reads the complete
# bloomberg_security_list.csv in its existing order and creates company workbooks
# containing at most 50 securities each, plus one macro workbook.
#
# The layouts and Bloomberg field definitions match excel_addin_formula_generator.py and
# generate_bloomberg_macro_workbook.py: row 1 contains paired Date/Value headers, BDH
# formulas start in row 2, and the cells below remain empty for Bloomberg results. Each
# company workbook starts with its own Security_List and then one sheet per security.
#
# Requires pandas and XlsxWriter in the notebook kernel. These files contain formulas;
# open them in Excel with the Bloomberg Add-In to retrieve data.


In [4]:
from pathlib import Path
import re

import pandas as pd
import xlsxwriter

# Locate the project root even when the kernel starts in notebooks/.
ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "data").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or a folder inside it.")

INPUT = ROOT / "data/intermediate/bloomberg_security_list.csv"
SECURITY_DIR = ROOT / "data/intermediate/security"
MACRO_DIR = ROOT / "data/intermediate/macro"
BATCH_SIZE = 50
# Keep False to protect workbooks already populated in Bloomberg.
OVERWRITE = True

print("Input:", INPUT)
print("Company workbooks:", SECURITY_DIR)
print("Macro workbook:", MACRO_DIR)


Input: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\intermediate\bloomberg_security_list.csv
Company workbooks: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\intermediate\security
Macro workbook: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\intermediate\macro


In [ ]:
# Load and validate the complete security list
#
# Every input record is included exactly once. Invalid identifiers, duplicate company
# IDs, or invalid date windows stop generation instead of silently dropping records. The
# final part can contain fewer than 50 securities.


In [5]:
companies = pd.read_csv(INPUT, dtype="string")
required = {"company_id", "company_type", "name", "bloomberg_identifier", "start_date", "end_date"}
missing = required - set(companies.columns)
if missing:
    raise ValueError(f"Security list is missing columns: {sorted(missing)}")
if companies.empty:
    raise ValueError("The security list is empty.")
if not isinstance(BATCH_SIZE, int) or isinstance(BATCH_SIZE, bool) or BATCH_SIZE <= 0:
    raise ValueError("BATCH_SIZE must be a positive integer.")

for column in ["company_id", "bloomberg_identifier"]:
    companies[column] = companies[column].str.strip()
    invalid = companies[column].fillna("").eq("")
    if column == "bloomberg_identifier":
        invalid |= companies[column].str.upper().eq("NOT_FOUND").fillna(False)
    if invalid.any():
        raise ValueError(f"{column} is missing or invalid at CSV rows {(companies.index[invalid] + 2).tolist()}.")
if companies["company_id"].duplicated().any():
    raise ValueError("company_id must be unique so every input record maps to one worksheet.")

date_windows = companies[["start_date", "end_date"]].apply(pd.to_datetime, errors="coerce")
invalid_dates = date_windows.isna().any(axis=1) | (date_windows["start_date"] > date_windows["end_date"])
if invalid_dates.any():
    raise ValueError(f"Invalid date windows at CSV rows {(companies.index[invalid_dates] + 2).tolist()}.")
for column in ["start_date", "end_date"]:
    companies[column] = date_windows[column].dt.strftime("%Y-%m-%d")

# One shared macro history covering every company's five-year collection window.
# Override these two values here if a different macro range is needed.
MACRO_START_DATE = date_windows["start_date"].min().strftime("%Y-%m-%d")
MACRO_END_DATE = date_windows["end_date"].max().strftime("%Y-%m-%d")

batches = [
    companies.iloc[start:start + BATCH_SIZE].copy()
    for start in range(0, len(companies), BATCH_SIZE)
]
security_paths = [
    SECURITY_DIR / f"bloomberg_addin_workbook_part_{part:03d}.xlsx"
    for part in range(1, len(batches) + 1)
]
macro_path = MACRO_DIR / "bloomberg_macro_addin_workbook.xlsx"
output_paths = [*security_paths, macro_path]
existing = [path for path in output_paths if path.exists()]
if existing and not OVERWRITE:
    raise FileExistsError(
        "Workbooks already exist. Set OVERWRITE = True only if they can be replaced:\n"
        + "\n".join(str(path) for path in existing)
    )

print(f"{len(companies)} securities -> {len(batches)} company workbooks")
print("Securities per part:", [len(batch) for batch in batches])
print(f"Macro date window: {MACRO_START_DATE} to {MACRO_END_DATE}")


538 securities -> 11 company workbooks
Securities per part: [50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 38]
Macro date window: 1993-06-18 to 2026-05-02


In [7]:
# Bloomberg fields
#
# These mappings and periodicities are copied from the existing generators. Company
# fundamentals use annual frequency, market fields use monthly frequency, and macro
# series retain their existing frequencies.


In [8]:
ANNUAL_FIELDS = {
    # Balance sheet
    "total_assets": "BS_TOT_ASSET",
    "total_liabilities": "BS_TOT_LIAB2",
    "total_debt": "SHORT_AND_LONG_TERM_DEBT",
    "short_term_debt": "BS_ST_BORROW",                 
    "long_term_debt": "BS_LT_BORROW",                  
    "cash_short_term_investments": "BS_CASH_NEAR_CASH_ITEM",
    "current_assets": "BS_CUR_ASSET_REPORT",
    "current_liabilities": "BS_CUR_LIAB",
    "shareholders_equity": "BS_SHAREHOLDER_EQY_ENDING_PERIOD",
    "accounts_receivable": "BS_ACCT_NOTE_RCV",         
    "inventory": "BS_INVENTORIES",                     
    "working_capital": "WORKING_CAPITAL",   

    # Income statement
    "revenue": "SALES_REV_TURN",
    "ebit": "EBIT",
    "ebitda": "EBITDA",
    "net_income": "NET_INCOME",
    "interest_expense": "IS_INT_EXPENSE",
    "depreciation_amortization": "IS_DEPR_EXP",     

    # Cash flow statement
    "operating_cash_flow": "CF_CASH_FROM_OPER",
    "capital_expenditures": "CAPITAL_EXPEND",
    "free_cash_flow": "CF_FREE_CASH_FLOW",            
}

MARKET_FIELDS = {
    "adjusted_price": "PX_LAST",
    "market_cap": "CUR_MKT_CAP",                    
    "shares_outstanding": "EQY_SH_OUT",
    "beta": "BETA_RAW_OVERRIABLE",                                 
    "volume": "PX_VOLUME",
    "turnover": "TURNOVER",                       
    "bid_ask_spread": "SPREAD_BA_CR",       
}

MACRO_FIELDS = {
    "inflation_cpi_yoy": {
        "security": "CPI YOY Index",
        "field": "PX_LAST",
        "periodicity": "M",
    },

    "real_gdp_growth": {
        "security": "GDP CQOQ Index",
        "field": "PX_LAST",
        "periodicity": "Q",
    },

    "unemployment_rate": {
        "security": "USURTOT Index",
        "field": "PX_LAST",
        "periodicity": "M",
    },

    "effective_federal_funds_rate": {
        "security": "FEDL01 Index",
        "field": "PX_LAST",
        "periodicity": "D",
    },

    "treasury_3m_rate": {
        "security": "USGG3M Index",
        "field": "PX_LAST",
        "periodicity": "D",
    },

    "treasury_10y_rate": {
        "security": "USGG10YR Index",
        "field": "PX_LAST",
        "periodicity": "D",
    },

    "credit_spread": {
        "security": "LUACTRUU Index",
        "field": "PX_LAST",
        "periodicity": "M",
    },

    "vix": {
        "security": "VIX Index",
        "field": "PX_LAST",
        "periodicity": "D",
    },
}


In [9]:
def safe_sheet_name(company_id, name, used_names):
    name = "" if pd.isna(name) else str(name)
    raw = re.sub(r"[\\/*?:\[\]]", "", f"{company_id}_{name}")[:31].strip("'") or str(company_id)
    candidate = raw
    counter = 2
    while candidate.casefold() in used_names:
        suffix = f"_{counter}"
        candidate = raw[:31 - len(suffix)] + suffix
        counter += 1
    used_names.add(candidate.casefold())
    return candidate


def bdh_formula(security, field, start_date, end_date, periodicity):
    arguments = [security, field, start_date, end_date, f"Per={periodicity}", "Days=T", "Dir=V"]
    return "=BDH(" + ",".join('"' + str(value).replace('"', '""') + '"' for value in arguments) + ")"


def write_field_block(worksheet, start_col, alias, field, security, start_date, end_date, periodicity, width=16):
    worksheet.write_string(0, start_col, f"{alias}_Date")
    worksheet.write_string(0, start_col + 1, f"{alias}_Value")
    worksheet.write_formula(1, start_col, bdh_formula(security, field, start_date, end_date, periodicity))
    worksheet.set_column(start_col, start_col + 1, width)


def write_company_workbook(batch, output):
    with xlsxwriter.Workbook(output) as workbook:
        index = workbook.add_worksheet("Security_List")
        index.write_row(0, 0, list(batch.columns))
        for row_number, values in enumerate(batch.itertuples(index=False, name=None), start=1):
            for column, value in enumerate(values):
                index.write_string(row_number, column, "" if pd.isna(value) else str(value))
        index.set_column(0, len(batch.columns) - 1, 18)

        used_names = {"security_list"}
        for company in batch.itertuples(index=False):
            sheet = workbook.add_worksheet(safe_sheet_name(company.company_id, company.name, used_names))
            for i, (alias, field) in enumerate(ANNUAL_FIELDS.items()):
                write_field_block(sheet, i * 2, alias, field, company.bloomberg_identifier,
                                  company.start_date, company.end_date, "Y")
            market_offset = len(ANNUAL_FIELDS) * 2
            for i, (alias, field) in enumerate(MARKET_FIELDS.items()):
                write_field_block(sheet, market_offset + i * 2, alias, field, company.bloomberg_identifier,
                                  company.start_date, company.end_date, "M")


def write_macro_workbook(output):
    with xlsxwriter.Workbook(output) as workbook:
        sheet = workbook.add_worksheet("Macro_Data")
        for i, (alias, config) in enumerate(MACRO_FIELDS.items()):
            write_field_block(sheet, i * 2, alias, config["field"], config["security"],
                              MACRO_START_DATE, MACRO_END_DATE, config["periodicity"], width=20)


In [10]:
# Generate the workbooks
#
# Company files are saved in data/intermediate/security/; the shared macro file is saved
# in data/intermediate/macro/. Existing files are protected by default. To intentionally
# regenerate them, set OVERWRITE = True in the setup cell and rerun from the top.


In [11]:
# Repeat the overwrite check here so rerunning this cell also protects saved data.
existing = [path for path in output_paths if path.exists()]
if existing and not OVERWRITE:
    raise FileExistsError("Output workbooks already exist; set OVERWRITE = True to replace them.")

SECURITY_DIR.mkdir(parents=True, exist_ok=True)
MACRO_DIR.mkdir(parents=True, exist_ok=True)

summary = []
for part, (batch, output) in enumerate(zip(batches, security_paths), start=1):
    write_company_workbook(batch, output)
    summary.append({
        "part": part,
        "securities": len(batch),
        "first_company_id": batch.iloc[0]["company_id"],
        "last_company_id": batch.iloc[-1]["company_id"],
        "workbook": output.name,
    })
    print(f"Created {output.name}: {len(batch)} securities")

write_macro_workbook(macro_path)
print(f"Created {macro_path.name}: {len(MACRO_FIELDS)} macro series")

summary = pd.DataFrame(summary)
display(summary)


Created bloomberg_addin_workbook_part_001.xlsx: 50 securities
Created bloomberg_addin_workbook_part_002.xlsx: 50 securities
Created bloomberg_addin_workbook_part_003.xlsx: 50 securities
Created bloomberg_addin_workbook_part_004.xlsx: 50 securities
Created bloomberg_addin_workbook_part_005.xlsx: 50 securities
Created bloomberg_addin_workbook_part_006.xlsx: 50 securities
Created bloomberg_addin_workbook_part_007.xlsx: 50 securities
Created bloomberg_addin_workbook_part_008.xlsx: 50 securities
Created bloomberg_addin_workbook_part_009.xlsx: 50 securities
Created bloomberg_addin_workbook_part_010.xlsx: 50 securities
Created bloomberg_addin_workbook_part_011.xlsx: 38 securities
Created bloomberg_macro_addin_workbook.xlsx: 8 macro series


,part,securities,first_company_id,last_company_id,workbook
0,1,50,SA_222,SA_140,bloomberg_addin_workbook_part_001.xlsx
1,2,50,SA_105,SA_274,bloomberg_addin_workbook_part_002.xlsx
2,3,50,SA_64,SA_288,bloomberg_addin_workbook_part_003.xlsx
3,4,50,SA_112,SA_132,bloomberg_addin_workbook_part_004.xlsx
4,5,50,SA_242,SA_249,bloomberg_addin_workbook_part_005.xlsx
5,6,50,SA_293,NB_186,bloomberg_addin_workbook_part_006.xlsx
6,7,50,NB_3028,NB_2337,bloomberg_addin_workbook_part_007.xlsx
7,8,50,NB_386,NB_2608,bloomberg_addin_workbook_part_008.xlsx
8,9,50,NB_2813,NB_2580,bloomberg_addin_workbook_part_009.xlsx
9,10,50,NB_1360,NB_2729,bloomberg_addin_workbook_part_010.xlsx


In [12]:
# Check that batching includes every security once, in the original order.
combined = pd.concat(batches, ignore_index=True)
pd.testing.assert_frame_equal(combined, companies.reset_index(drop=True))
assert all(0 < len(batch) <= BATCH_SIZE for batch in batches)
assert all(len(batch) == BATCH_SIZE for batch in batches[:-1])
assert summary["securities"].sum() == len(companies)
assert all(path.is_file() and path.stat().st_size > 0 for path in output_paths)
print(f"Ready: {len(companies)} securities in {len(security_paths)} company workbooks, plus one macro workbook.")


Ready: 538 securities in 11 company workbooks, plus one macro workbook.


In [13]:
# Retrieve the Bloomberg data
#
# 1. Open each company part and the macro workbook in Excel with the Bloomberg Add-In
# active.
# 2. Refresh the BDH formulas and allow each history to fill its Date/Value columns.
# 3. Save the populated workbooks. Preserve the worksheet names and paired column
# headers for later cleaning.
